# 04 — Feature engineering and selection

**Question:** which parts of the 21-second context window actually carry signal?

The features were engineered by the study team before this extract: per-second summary statistics, ±10-second lags and leads, and aggregates over those windows. Rather than invent more, this notebook tests which existing blocks matter — an ablation, not an expansion.

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
from smartnet import config
from smartnet.data import loader
from smartnet.models.experiment import run_cv

df = loader.load_analysis_frame()
for b, c in config.FEATURE_BLOCKS.items():
    print(f'{b:<26} {len(c):3d}')

## Ablation

Each block is evaluated under grouped-event CV with the same random forest. If the 40 individual lag columns add nothing over the 6 aggregates, the feature set can be cut by 85% with no cost — which matters for on-device inference.

In [ ]:
rows = []
for block in config.FEATURE_BLOCKS:
    r = run_cv(df, 'motion_5cat', 'random_forest', 'grouped_event',
               features=config.FEATURE_BLOCKS[block])
    rows.append({'block': block, 'n_features': r['n_features'],
                 'accuracy': round(r['cv_accuracy_mean'], 4),
                 'balanced_accuracy': round(r['cv_balanced_accuracy_mean'], 4),
                 'macro_f1': round(r['cv_macro_f1_mean'], 4)})
ablation = pd.DataFrame(rows).sort_values('balanced_accuracy', ascending=False)
ablation

In [ ]:
fig, ax = plt.subplots(figsize=(7,3.2))
ax.barh(ablation.block, ablation.balanced_accuracy, color='#17365D')
for i,(b,v,n) in enumerate(zip(ablation.block, ablation.balanced_accuracy, ablation.n_features)):
    ax.text(v+0.005, i, f'{v:.3f}  ({n} features)', va='center', fontsize=8)
ax.set_xlim(0,1.05); ax.set_xlabel('Balanced accuracy (grouped-event CV)')
ax.set_title('Feature block ablation'); ax.grid(axis='y', visible=False); plt.show()

## Why no additional features were engineered

Frequency-domain features (FFT, spectral energy) were considered and rejected. They require the raw 10 Hz waveform; this extract contains only per-second summary statistics, so the underlying signal needed to compute them is not present. Claiming spectral features here would mean fabricating them.

Hour-of-day was excluded deliberately: notebook 03 shows motions were staged in daytime sessions, so time of day would separate the classes for reasons that have nothing to do with movement and would not survive deployment.